# 3/ Lakehouse Applications로 프론트엔드 앱 배포하기


Mosaic AI Agent Evaluation 리뷰 앱은 개발 과정에서 이해관계자의 피드백을 수집하는 데 사용됩니다.

하지만 여전히 자체 프론트엔드 애플리케이션을 배포해야 합니다!

Databricks Lakehouse Applications를 활용하여 첫 번째 간단한 챗봇 프론트엔드 앱을 빌드하고 배포해 보겠습니다.

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/chatbot-rag/rag-frontend-app.png?raw=true" width="1200px">


<div style="background-color: #d4e7ff; padding: 10px; border-radius: 15px;">
<strong>참고:</strong> 이 예제에서는 엔드포인트를 사용하여 앱을 배포합니다. 하지만 앱 자체만 사용하는 경우, MLFlow Chat Agent를 애플리케이션 내에 직접 패키징하고 엔드포인트를 완전히 제거할 수도 있습니다!
</div>

<!-- Collect usage data (view). Remove it to disable collection or disable tracker during installation. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=data-science&org_id=601859075515029&notebook=%2F04-deploy-app%2F04-Deploy-Frontend-Lakehouse-App&demo_name=ai-agent&event=VIEW&path=%2F_dbdemos%2Fdata-science%2Fai-agent%2F04-deploy-app%2F04-Deploy-Frontend-Lakehouse-App&version=1">

In [0]:
%pip install --quiet -U mlflow[databricks]>=3.1.4 databricks-sdk>=0.59.0
dbutils.library.restartPython()

In [0]:
%run ../_resources/01-setup

## 애플리케이션 설정 추가하기

Lakehouse 앱은 모든 Python 프레임워크와 호환됩니다. 이 데모에서는 모델 서빙 엔드포인트 이름을 포함하는 간단한 설정 파일을 생성하여 `chatbot_app/app.yaml`로 저장하겠습니다.

In [0]:
print(f"The Databricks APP will be using the following model serving endpoint: {ENDPOINT_NAME}")

In [0]:
import yaml

# 프론트엔드 애플리케이션이 배포한 모델 엔드포인트에 접근합니다.
# dbdemos는 카탈로그와 데이터베이스를 변경할 수 있으므로, 적절한 엔드포인트 이름으로 앱을 배포해야 합니다.
yaml_app_config = {"command": ["uvicorn", "main:app", "--workers", "1"],
                    "env": [{"name": "MODEL_SERVING_ENDPOINT", "value": ENDPOINT_NAME}]
                  }
try:
    with open('chatbot_app/app.yaml', 'w') as f:
        yaml.dump(yaml_app_config, f)
except Exception as e:
    print(f'빌드 작업에서 통과 - {e}')

## MLFlow Tracing과 Feedback API를 통한 피드백 수집

MLFlow 3을 사용하면 애플리케이션에서 피드백(좋아요/싫어요)을 직접 수집하는 것이 이제 쉽습니다!

```
client = mlflow.deployments.get_deploy_client("databricks")

input_message = [{"content": "test", "role": "user", "type": "message"}]

response = client.predict(
  endpoint=ENDPOINT_NAME,
  inputs={'input': input_message, "databricks_options": {
      # 피드백 로깅을 위해 trace_id를 가져올 수 있도록 trace를 반환합니다. (빠른 결과를 위해 id만 반환)
      "return_trace": True
    }}
)
```


그런 다음 챗봇 백엔드에서 `mlflow-tracing`을 사용하여 사용자 피드백과 함께 trace를 전송하기만 하면 됩니다:


```
mlflow.log_feedback(
                trace_id=trace_id, # 응답에 포함된 trace id, 일반적으로 tr-xxxxx 형식
                name='user_feedback',
                value=True if like_data.liked else False,
                rationale=None,
                source=mlflow.entities.AssessmentSource(source_type='HUMAN', source_id='user')
            )
```

*참고: 자체 ID도 관리할 수 있습니다 - 자세한 내용은 [피드백 문서](https://docs.databricks.com/aws/en/mlflow3/genai/tracing/collect-user-feedback/)를 참조하세요*

## 이제 Databricks Applications를 사용하여 Gradio로 챗봇 애플리케이션을 만들어 보겠습니다

## 애플리케이션 배포하기

애플리케이션은 `chatbot_app` 폴더 내의 2개 파일로 구성됩니다:
- `main.py`: Python 코드가 포함되어 있습니다
- `app.yaml`: 설정이 포함되어 있습니다

이제 API를 호출하여 새 앱을 만들고 `chatbot_app` 경로를 사용하여 배포하기만 하면 됩니다:

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.apps import App, AppResource, AppResourceServingEndpoint, AppResourceServingEndpointServingEndpointPermission, AppDeployment

w = WorkspaceClient()
app_name = "dbdemos-ai-agent-app"

Lakehouse 앱에는 자동 프로비저닝된 Service Principal이 제공됩니다. 배포하기 전에 이 Service Principal에 모델 엔드포인트에 대한 액세스 권한을 부여하겠습니다...

In [0]:
serving_endpoint = AppResourceServingEndpoint(name=ENDPOINT_NAME,
                                              permission=AppResourceServingEndpointServingEndpointPermission.CAN_QUERY
                                              )

rag_endpoint = AppResource(name="rag-endpoint", serving_endpoint=serving_endpoint) 

rag_app = App(name=app_name, 
              description="Your Databricks assistant", 
              default_source_code_path=os.path.join(os.getcwd(), 'chatbot_app'),
              resources=[rag_endpoint])
try:
  app_details = w.apps.create_and_wait(app=rag_app)
  print(app_details)
except Exception as e:
  if "already exists" in str(e):
    print("앱이 이미 존재합니다. 배포할 수 있습니다")
  else:
    raise e

앱이 생성되면 다음과 같이 코드를 (재)배포할 수 있습니다:

In [0]:
import mlflow

xp_name = os.getcwd().rsplit("/", 1)[0]+"/03-knowledge-base-rag/03.1-pdf-rag-tool"
mlflow.set_experiment(xp_name)

In [0]:
deployment = AppDeployment(source_code_path=os.path.join(os.getcwd(), 'chatbot_app'))

app_details = w.apps.deploy_and_wait(app_name=app_name, app_deployment=deployment)

In [0]:
# 애플리케이션에 접속해 봅시다
w.apps.get(name=app_name).url

## Lakehouse 앱이 준비되어 배포되었습니다!

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/chatbot-rag/rag-gradio-app.png?raw=true" width="750px" style="float: right; margin-left:10px">

UI를 열어 챗봇에 요청을 시작하세요.

개선 사항으로, 챗봇 UI를 향상하여 피드백을 제공하고 Mosaic AI Quality Labs로 전송할 수 있습니다. 그러면 잘못된 답변을 검토하고 개선할 수 있습니다.

## 결론

Databricks가 다음과 같은 엔드투엔드 플랫폼을 제공하는 것을 확인했습니다:
- 엔드포인트 구축 및 배포
- 챗봇을 검토, 분석 및 개선하기 위한 내장 솔루션
- Lakehouse 앱으로 GenAI 프론트엔드 애플리케이션 배포!

## 다음 단계: 한 단계 더 나아갈 준비가 되셨나요? 프로덕션 환경에서 에이전트 성능을 모니터링해 봅시다

[05-production-monitoring/05.production-monitoring]($../05-production-monitoring/05.production-monitoring) 노트북을 열어 프로덕션 환경에서 엔드포인트를 모니터링하고 평가하는 방법을 알아보세요!